In [ ]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel

In [ ]:
mode_path = '../source/lesson_models/Qwen3-8B'
lora_path = './outputs/Qwen3-8B/checkpoint-351'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(mode_path, use_fast=False, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(mode_path, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)

model = PeftModel.from_pretrained(model, model_id=lora_path)

In [ ]:
print('\U0001F60D哈喽，我是基于Qwen3-8B微调的甄嬛体问答模型，我可以用甄嬛体来回答你的问题哦~')
while True:
    prompt = input('\U0001F600我是嬛嬛，你请说：')
    if prompt == 'exit':
        print('\U0001F62D好的拜拜，欢迎再次使用~')
        break

    inputs = tokenizer.apply_chat_template(
                                        [{"role": "user", "content": "假设你是皇帝身边的女人--甄嬛。"},{"role": "user", "content": prompt}],
                                        add_generation_prompt=True,
                                        tokenize=True,
                                        return_tensors="pt",
                                        return_dict=True,
                                        enable_thinking=False
                                    )


    gen_kwargs = {"max_length": 2500, "do_sample": True, "top_k": 1}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
        print(f"\U0001F60DAI嬛儿的回复：\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")